# CycleGAN บน Vast P100

กด **Run → Run All Cells** ได้เลย ระบบดาวน์โหลดข้อมูล ติดตั้ง และรัน experiment ให้เอง

ค่าเริ่มต้นคือ **JC-base** ชุดเดิมบน Kaggle P100 (30 epochs) เมื่อเสร็จจะมีลิงก์ดาวน์โหลดผลด้านล่าง

**อยากปรับค่าเอง:** ใส่ค่าที่ต้องการใน `SETTINGS` ด้านล่าง เช่น epochs, batch_size, learning_rate, sigma_d, seed
เว้น `SETTINGS = {}` เพื่อใช้ค่าเดิม Custom run จะแยกโฟลเดอร์ผลและเทรน classifier ใหม่ด้วย
ตั้งค่าเดิมแล้ว Run All อีกครั้ง = ทำงานต่อ; เปลี่ยนค่า = เริ่มรันใหม่

ถ้าครบเวลารอบละ 10 ชั่วโมงแล้วยังไม่เสร็จ กด Run All อีกครั้งเพื่อทำต่อบนเครื่องเดิม
ดาวน์โหลดผลก่อนเลิกใช้เครื่อง และกด Stop ใน Vast เมื่อใช้งานเสร็จ

<details><summary>เลือก experiment อื่น</summary>

เปลี่ยนชื่อในช่องด้านล่างเป็น JC-base, JC-partial, JC-full, CP-base, CP-partial, CP-full, JP-base, JP-partial หรือ JP-full แล้วกด Run All

เมื่อ `SETTINGS = {}`: JC-base เริ่มเทรนใหม่ ส่วนอีก 8 ตัวใช้ชุดข้อมูลและ JC classifier เดิมจากการทดลอง Kaggle
</details>


In [ ]:
EXPERIMENT = "JC-base"
SETTINGS = {}
# ตัวอย่าง: SETTINGS = {"epochs": 50, "batch_size": 8, "learning_rate": 0.0001, "sigma_d": 0.2, "seed": 42}


In [ ]:
CUSTOM_RUNNER_SOURCE = '"""Custom schedules around the frozen TF1.4 engines; modern Python only."""\nimport argparse\nimport hashlib\nimport importlib\nimport json\nimport math\nfrom pathlib import Path\nimport shutil\nimport sys\nimport time\n\n\nFIELDS = {\n    \'epochs\', \'classifier_epochs\', \'batch_size\', \'learning_rate\', \'sigma_d\',\n    \'seed\', \'lambda_cycle\', \'ngf\', \'ndf\', \'decay_start_epoch\', \'evaluate_epochs\',\n    \'beta1\', \'beta2\',\n}\n\n\ndef validate_settings(settings):\n    if not isinstance(settings, dict):\n        raise ValueError(\'SETTINGS ต้องเป็น dictionary เช่น {"epochs": 50}\')\n    unknown = set(settings) - FIELDS\n    if unknown:\n        raise ValueError(\'ไม่รองรับ SETTINGS: \' + \', \'.join(sorted(map(str, unknown))))\n\n\ndef configuration(bundle, experiment, settings):\n    validate_settings(settings)\n    path = bundle / \'kaggle\' / (\'config.json\' if experiment == \'JC-base\'\n                               else \'configs/\' + experiment + \'.json\')\n    config = json.loads(path.read_text())\n    config.update(settings)\n    for key in (\'epochs\', \'classifier_epochs\', \'batch_size\', \'ngf\', \'ndf\'):\n        if type(config[key]) is not int or config[key] < 1:\n            raise ValueError(key + \' ต้องเป็นจำนวนเต็มมากกว่า 0\')\n    if type(config[\'seed\']) is not int or not 0 <= config[\'seed\'] < 2**31:\n        raise ValueError(\'seed ต้องเป็นจำนวนเต็มตั้งแต่ 0 ถึง 2147483647\')\n    for key in (\'learning_rate\', \'lambda_cycle\', \'sigma_d\', \'beta1\', \'beta2\'):\n        value = config[key]\n        if type(value) not in (float, int) or not math.isfinite(value):\n            raise ValueError(key + \' ต้องเป็นตัวเลขที่มีค่าจำกัด\')\n        minimum_ok = value >= 0 if key in (\'sigma_d\', \'beta1\', \'beta2\') else value > 0\n        if not minimum_ok or (key in (\'beta1\', \'beta2\') and value >= 1):\n            raise ValueError(\'ค่า \' + key + \' อยู่นอกช่วงที่ใช้ได้\')\n    shortest = min(config[\'epochs\'], config[\'classifier_epochs\'])\n    if \'decay_start_epoch\' not in settings:\n        config[\'decay_start_epoch\'] = min(config[\'decay_start_epoch\'], shortest // 3)\n    decay = config[\'decay_start_epoch\']\n    if type(decay) is not int or not 0 <= decay < shortest:\n        raise ValueError(\'decay_start_epoch ต้องตั้งแต่ 0 และน้อยกว่า epochs กับ classifier_epochs\')\n    epochs = settings.get(\'evaluate_epochs\', [e for e in config[\'evaluate_epochs\'] if e <= config[\'epochs\']])\n    if not isinstance(epochs, list) or any(type(e) is not int or not 1 <= e <= config[\'epochs\'] for e in epochs):\n        raise ValueError(\'evaluate_epochs ต้องเป็นรายการ epoch ตั้งแต่ 1 ถึง epochs\')\n    config[\'evaluate_epochs\'] = sorted(set(epochs + [config[\'epochs\']]))\n    if len(config[\'evaluate_epochs\']) > 30:\n        raise ValueError(\'เลือก evaluate_epochs ได้ไม่เกิน 30 ค่า รวม epoch สุดท้าย\')\n    config[\'protocol\'] = \'native-tf14-custom-v1\'\n    config[\'paper_reference\'] = None\n    return config\n\n\ndef request_id(request):\n    raw = json.dumps(request, sort_keys=True, separators=(\',\', \':\'), allow_nan=False)\n    return hashlib.sha256(raw.encode()).hexdigest()[:16]\n\n\ndef expected_steps(config, manifest, kind):\n    a, b = config.get(\'dataset_a\', \'JC_J\'), config.get(\'dataset_b\', \'JC_C\')\n    counts = [manifest[\'splits\'][key + \'_train\'][\'count\'] for key in (a, b)]\n    examples = min(counts) if kind == \'gan\' else sum(counts)\n    batches = examples // config[\'batch_size\']\n    if not batches:\n        raise ValueError(\'batch_size ใหญ่กว่าจำนวนข้อมูลสำหรับ \' + kind)\n    return batches * config[\'epochs\' if kind == \'gan\' else \'classifier_epochs\']\n\n\ndef stage_complete(stage, config, manifest, kind):\n    marker = stage / \'complete.json\'\n    if not marker.exists():\n        return False\n    result = json.loads(marker.read_text())\n    expected = expected_steps(config, manifest, kind)\n    epoch = config[\'epochs\' if kind == \'gan\' else \'classifier_epochs\']\n    checkpoint = stage / (\'epoch-{:02d}\'.format(epoch)) / (\'model-\' + str(expected))\n    if result.get(\'state\') != \'complete\' or result.get(\'completed_steps\') != expected:\n        raise ValueError(\'จำนวน steps ไม่ตรงกับ custom config: \' + kind)\n    for suffix in (\'.index\', \'.data-00000-of-00001\'):\n        path = Path(str(checkpoint) + suffix)\n        if not path.is_file() or not path.stat().st_size:\n            raise ValueError(\'ไม่พบ checkpoint สุดท้ายของ \' + kind)\n    return True\n\n\ndef run(args):\n    bundle, root = args.bundle.resolve(), args.output.resolve()\n    request = json.loads(args.request.read_text())\n    config = configuration(bundle, request[\'experiment\'], request[\'settings\'])\n    if config != request[\'config\']:\n        raise ValueError(\'Custom config changed after launch\')\n    runner_bytes = Path(__file__).read_bytes()\n    if hashlib.sha256(runner_bytes).hexdigest() != request[\'runner_sha256\']:\n        raise ValueError(\'Custom runner changed after launch\')\n    previous = root / \'custom-run.json\'\n    if previous.exists() and json.loads(previous.read_text()) != request:\n        raise ValueError(\'Resume requires the same custom settings and source\')\n    sys.path.insert(0, str(bundle / \'kaggle\'))\n    helper = importlib.import_module(\'session\' if request[\'experiment\'] == \'JC-base\' else \'suite_session\')\n    helper.prepare_run(bundle, root)\n    helper.save(previous, request)\n    helper.save(root / \'custom-config.json\', config)\n    (root / \'custom-runner.py\').write_bytes(runner_bytes)\n    manifest = json.loads((root / \'data/manifest.json\').read_text())\n    env = helper.runtime_environment(args.python.resolve().parent.parent)\n    deadline = time.monotonic() + max(0, args.seconds - 900)\n    status = dict(state=\'starting\', protocol=config[\'protocol\'], experiment=request[\'experiment\'],\n                  epochs=config[\'epochs\'], classifier_epochs=config[\'classifier_epochs\'])\n    archive = root.parent / (root.parent.name + \'-results.zip\')\n\n    def publish(**values):\n        status.update(values)\n        helper.save(root / \'session-status.json\', status)\n\n    common = [str(args.python), str(root / \'code/experiment.py\'), \'--config\',\n              str(root / \'custom-config.json\'), \'--data\', str(root / \'data\'), \'--device\', \'gpu\']\n    try:\n        for kind in (\'gan\', \'classifier\'):\n            expected_steps(config, manifest, kind)\n        publish(phase=\'preflight\')\n        if not helper.execute([str(args.python), str(root / \'code/kaggle/preflight.py\'), \'--device\', \'gpu\',\n                               \'--report\', str(root / \'preflight.json\')], root / \'preflight.log\', env, deadline):\n            publish(state=\'paused\', reason=\'session_budget\')\n            return\n        for kind in (\'gan\', \'classifier\'):\n            stage = root / kind\n            if stage_complete(stage, config, manifest, kind):\n                continue\n            remaining = deadline - time.monotonic()\n            if remaining < 300:\n                publish(state=\'paused\', reason=\'session_budget\')\n                return\n            if shutil.disk_usage(root).free < 3 * 1024**3:\n                publish(state=\'paused\', reason=\'disk_space_less_than_3GiB\')\n                return\n            command = common + [\'train\', \'--kind\', kind, \'--run-dir\', str(stage),\n                                \'--max-seconds\', str(remaining - 180)]\n            if stage.exists():\n                if (stage / \'checkpoints/checkpoint\').exists():\n                    command.append(\'--resume\')\n                else:\n                    stage.rename(stage.with_name(stage.name + \'-interrupted-\' + str(time.time_ns())))\n            publish(state=\'running\', phase=kind)\n            ready = helper.execute(command, root / (kind + \'.log\'), env, deadline, stage / \'status.json\')\n            if not ready or not stage_complete(stage, config, manifest, kind):\n                publish(state=\'paused\', reason=\'session_budget\')\n                return\n        reports = {}\n        for epoch in config[\'evaluate_epochs\']:\n            stage = root / (\'evaluation-epoch-{:02d}\'.format(epoch))\n            report_path = stage / \'report.json\'\n            if not report_path.exists():\n                if deadline - time.monotonic() < 600:\n                    publish(state=\'paused\', reason=\'session_budget\')\n                    return\n                if stage.exists():\n                    stage.rename(stage.with_name(stage.name + \'-interrupted-\' + str(time.time_ns())))\n                publish(state=\'running\', phase=\'evaluate\', epoch=epoch)\n                ready = helper.execute(common + [\'evaluate\', \'--run-dir\', str(stage),\n                    \'--gan-checkpoint\', str(root / \'gan\' / (\'epoch-{:02d}\'.format(epoch))),\n                    \'--classifier-checkpoint\', str(root / \'classifier\' /\n                                                  (\'epoch-{:02d}\'.format(config[\'classifier_epochs\'])))],\n                    root / (stage.name + \'.log\'), env, deadline)\n                if not ready:\n                    publish(state=\'paused\', reason=\'session_budget\')\n                    return\n            report = json.loads(report_path.read_text())\n            report.update(protocol=config[\'protocol\'], paper_reference=None,\n                          paper_caveat=\'Custom settings; classifier trained for this run. Not a registered paper result.\')\n            helper.save(report_path, report)\n            reports[str(epoch)] = report\n        helper.save(root / \'results.json\', dict(protocol=config[\'protocol\'], config=config, reports=reports,\n                    classifier=\'Trained from scratch for this custom run; no historical classifier reused.\'))\n        publish(state=\'complete\', phase=\'evaluated\')\n    except BaseException as error:\n        publish(state=\'failed\', error=repr(error))\n        raise\n    finally:\n        helper.export_results(root).replace(archive)\n        print(\'Saved custom results:\', archive, flush=True)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in (\'bundle\', \'python\', \'output\', \'request\'):\n        parser.add_argument(\'--\' + name, type=Path, required=True)\n    parser.add_argument(\'--seconds\', type=float, default=36000)\n    run(parser.parse_args())\n'

"""Browser notebook launcher for the unchanged native TF1.4 experiments."""
import hashlib
import json
from pathlib import Path
import platform
import subprocess
import sys
import time
import types
import urllib.request
import zipfile

RELEASE = "https://github.com/themes12/CycleGAN-Music-Style-Transfer/releases/download/kaggle-tf14-v1/"
EXPERIMENTS = tuple(pair + "-" + model for pair in ("JC", "CP", "JP")
                    for model in ("base", "partial", "full"))
ASSETS = {
    "base": ("cyclegan-kaggle-tf14.zip", "e0713d42f39d0ee59dd40b46c507e5e74cbaed87c2ced8bc62d7bb2301e3d88e"),
    "suite": ("cyclegan-paper-suite.zip", "b00ceb9896899c86b31900be8f386a3a02bfe102e8751425c2f685048b177219"),
}


def digest(path):
    result = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            result.update(block)
    return result.hexdigest()


def prepare_bundle(workspace, kind):
    name, expected = ASSETS[kind]
    archive = workspace / name
    if not archive.exists() or digest(archive) != expected:
        temporary = archive.with_suffix(".download")
        with urllib.request.urlopen(RELEASE + name, timeout=120) as response, temporary.open("wb") as output:
            while True:
                block = response.read(1024 * 1024)
                if not block:
                    break
                output.write(block)
        if digest(temporary) != expected:
            raise RuntimeError("ดาวน์โหลดไม่ครบ กรุณากด Run All อีกครั้ง")
        temporary.replace(archive)
    destination = workspace / ("input-" + kind)
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as package:
        for member in package.infolist():
            path = (destination / member.filename).resolve()
            if not path.is_relative_to(destination.resolve()) or "\\" in member.filename:
                raise ValueError("Unsafe archive path")
        package.extractall(destination)
    manifests = list(destination.rglob("bundle-manifest.json"))
    if len(manifests) != 1:
        raise RuntimeError("ไม่พบชุด experiment ในไฟล์ที่ดาวน์โหลด")
    return manifests[0].parent


def environment_path(workspace, bundle):
    locks = ("conda-linux-64.lock", "pip-bootstrap.lock", "requirements-tf14.lock")
    fingerprint = hashlib.sha256(b"".join((bundle / "kaggle" / name).read_bytes()
                                         for name in locks)).hexdigest()
    for number in range(1000):
        prefix = workspace / ("tf14-env-" + str(number))
        if not prefix.exists():
            return prefix
        try:
            marker = json.loads((prefix / "kaggle-runtime.json").read_text())
            if marker["lock_sha256"] == fingerprint:
                return prefix
        except (OSError, ValueError, KeyError):
            pass  # Preserve interrupted installations and use a fresh directory.
    raise RuntimeError("มีการติดตั้งที่ค้างจำนวนมาก กรุณาใช้โฟลเดอร์ notebook ใหม่")


def read_status(path):
    try:
        return json.loads(path.read_text())
    except (OSError, ValueError):
        return {}


def progress_text(root):
    status = read_status(root / "session-status.json")
    phase = status.get("phase", "")
    label = {"gan": "กำลังเทรน CycleGAN", "classifier": "กำลังเทรน classifier",
             "preflight": "กำลังเริ่ม GPU", "evaluate": "กำลังประเมินผล",
             "evaluated": "กำลังจัดไฟล์ผลลัพธ์", "finished": "กำลังจัดไฟล์ผลลัพธ์"}.get(phase, "กำลังรัน")
    if phase.startswith("evaluation-"):
        label = "กำลังประเมินผล"
    if phase in ("gan", "classifier"):
        detail = read_status(root / status.get("stage", phase) / "status.json")
        if "epoch" in detail:
            total = status.get("classifier_epochs" if phase == "classifier" else "epochs", 30)
            label += " · epoch " + str(detail["epoch"]) + "/" + str(total)
    return label


def execute(command, log, message, update, lock_fd, root=None):
    started = time.monotonic()
    with log.open("w", encoding="utf-8") as output:
        process = subprocess.Popen(command, stdout=output, stderr=subprocess.STDOUT,
                                   pass_fds=(lock_fd,))
        try:
            while process.poll() is None:
                label = progress_text(root) if root else message
                update(label + " · " + str(int((time.monotonic() - started) / 60)) + " นาที")
                time.sleep(5)
        except BaseException:
            if process.poll() is None:
                process.terminate()
                update("กำลังหยุดงานและจัดเก็บผล กรุณารอสักครู่")
                process.wait()
            raise
    if process.returncode:
        raise RuntimeError("งานหยุดระหว่างรัน รายละเอียดอยู่ใน " + log.name)


def load_custom(source=None):
    if source is None:
        source = Path(__file__).with_name("vast_custom.py").read_text(encoding="utf-8")
    module = types.ModuleType("vast_custom")
    exec(compile(source, "vast_custom.py", "exec"), module.__dict__)
    return module, source


def training_command(experiment, workspace, bundle, prefix, settings=None, custom_source=None):
    if experiment not in EXPERIMENTS:
        raise ValueError("ชื่อ experiment ไม่ถูกต้อง: " + experiment)
    base = experiment == "JC-base"
    if settings:
        custom, source = load_custom(custom_source)
        request = dict(experiment=experiment, settings=settings,
                       config=custom.configuration(bundle, experiment, settings),
                       bundle_sha256=ASSETS["base" if base else "suite"][1],
                       runner_sha256=hashlib.sha256(source.encode("utf-8")).hexdigest())
        identifier = custom.request_id(request)
        directory = workspace / "results/custom" / (experiment + "-" + identifier)
        directory.mkdir(parents=True, exist_ok=True)
        runner = directory / "custom-session.py"
        runner.write_bytes(source.encode("utf-8"))
        request_path = directory / "request.json"
        request_path.write_text(json.dumps(request, indent=2), encoding="utf-8")
        root = directory / "run"
        archive = directory / (directory.name + "-results.zip")
        return ([sys.executable, "-u", str(runner), "--bundle", str(bundle),
                 "--python", str(prefix / "bin/python"), "--output", str(root),
                 "--request", str(request_path), "--seconds", "36000"], root, archive)
    root = workspace / "results" / ("base" if base else "suite") / "run"
    script = "session.py" if base else "suite_session.py"
    archive = root.parent / ("tf14-results.zip" if base else "paper-suite-results.zip")
    command = [sys.executable, "-u", str(bundle / "kaggle" / script),
               "--bundle", str(bundle), "--python", str(prefix / "bin/python"),
               "--output", str(root), "--mode", "train", "--device", "gpu", "--seconds", "36000"]
    if not base:
        command += ["--runs", experiment]
    return command, root, archive


def start(experiment, settings=None, custom_source=None):
    from IPython.display import FileLink, display
    notice = display("กำลังเตรียม " + experiment, display_id=True)
    update = notice.update
    workspace = Path.cwd().resolve() / "cyclegan-vast"
    workspace.mkdir(exist_ok=True)
    log = None
    archive = None

    def link(path, label):
        if path and path.exists():
            print(label)
            display(FileLink(path.relative_to(Path.cwd().resolve()).as_posix()))

    try:
        settings = {} if settings is None else settings
        custom, custom_source = load_custom(custom_source)
        custom.validate_settings(settings)
        if experiment not in EXPERIMENTS:
            raise ValueError("เลือกชื่อ experiment จากรายการด้านบน")
        if platform.system() != "Linux" or platform.machine() != "x86_64":
            raise RuntimeError("เปิด notebook นี้ใน Jupyter บนเครื่อง Vast P100")
        import fcntl
        with (workspace / ".launcher.lock").open("a") as lock:
            try:
                fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
            except BlockingIOError:
                raise RuntimeError("มีงานกำลังรันอยู่แล้ว ให้รอ notebook เดิมจนเสร็จ")
            devices = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True)
            if "P100" not in devices:
                raise RuntimeError("ชุด experiment เดิมนี้ใช้ Tesla P100 กรุณาเลือกเครื่อง P100")
            update("กำลังดาวน์โหลดโค้ดและข้อมูล " + experiment)
            bundle = prepare_bundle(workspace, "base" if experiment == "JC-base" else "suite")
            prefix = environment_path(workspace, bundle)
            command, root, archive = training_command(experiment, workspace, bundle, prefix, settings, custom_source)
            log = workspace / "setup.log"
            execute([sys.executable, str(bundle / "kaggle/bootstrap.py"), "--prefix", str(prefix)],
                    log, "กำลังติดตั้ง ครั้งแรกอาจใช้หลายนาที", update, lock.fileno())
            root.parent.mkdir(parents=True, exist_ok=True)
            log = root.parent / "session.log"
            execute(command, log, "กำลังเริ่มงาน", update, lock.fileno(), root)
            status = read_status(root / "session-status.json")
            if status.get("state") in ("complete", "selection_complete"):
                update(experiment + " เสร็จแล้ว ดาวน์โหลดผลด้านล่างได้เลย")
            elif status.get("state") == "paused":
                if status.get("reason") == "disk_space_less_than_3GiB":
                    update("พักงานเพราะพื้นที่ใกล้เต็ม ดาวน์โหลดผลที่บันทึกไว้ด้านล่าง")
                else:
                    update("บันทึกงานไว้แล้ว ครบเวลารอบนี้ 10 ชั่วโมง กด Run All อีกครั้งเพื่อทำต่อ")
            else:
                raise RuntimeError("งานยังไม่เสร็จ ดูรายละเอียดในไฟล์ log ด้านล่าง")
            link(archive, "ดาวน์โหลดผล (.zip):")
    except KeyboardInterrupt:
        update("หยุดงานแล้ว")
        link(archive, "ผลที่จัดเก็บล่าสุด (.zip):")
        link(log, "รายละเอียดงาน:")
    except Exception as error:
        update(str(error))
        link(log, "ดาวน์โหลดรายละเอียดข้อผิดพลาด:")
        link(archive, "ผลที่จัดเก็บล่าสุด (.zip):")


start(EXPERIMENT, SETTINGS, custom_source=CUSTOM_RUNNER_SOURCE)
